In [0]:
dbutuls.notebook.run(./create_table_utillity)

#### Reading flight data using autoloader

In [0]:
spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")
from pyspark.sql.functions import col, expr, count
# Reading data
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/dbfs/FileStore/tables/schema/flight")
    .load("/mnt/geeks/rw_adls/flight/")
)
display(df.dtypes)
# from pyspark.sql.functions import col

# # Regular expression for valid HHMM-style integers (e.g., "0000" to "2359")
# valid_time_pattern = "^[0-9]{1,4}$"

# # Filter rows where ANY of the time columns are NOT valid
# df_invalid_times = df.filter(col("DepTime").rlike(valid_time_pattern))
# display(df_invalid_times)

display(df)


In [0]:
from pyspark.sql.functions import to_date, current_timestamp, concat_ws, expr

df = df.withColumn("Date_Part", to_date(current_timestamp()))

df_base = df.selectExpr(
    "to_date(concat_ws('-',year,month,dayofmonth),'yyyy-MM-dd') as date",
    """
       date_format( try_to_timestamp( lpad(
                    CASE 
					    WHEN DepTime = 'NA' THEN NULL
                        WHEN DepTime = '2400' THEN '0000'
                        ELSE DepTime 
					END, 
		4, '0' ),'HHmm'), 'HH:mm') as deptime
    """,
    """
       date_format( try_to_timestamp( lpad(
                    CASE 
					    WHEN CRSDepTime = 'NA' THEN NULL
                        WHEN CRSDepTime = '2400' THEN '0000'
                        ELSE CRSDepTime 
					END, 
		4, '0' ),'HHmm'), 'HH:mm') as CRSDepTime
    """,
    """
       date_format( try_to_timestamp( lpad(
                    CASE 
					    WHEN ArrTime = 'NA' THEN NULL
                        WHEN ArrTime = '2400' THEN '0000'
                        ELSE ArrTime 
					END, 
		4, '0' ),'HHmm'), 'HH:mm') as ArrTime
    """,
    """
       date_format( try_to_timestamp( lpad(
                    CASE 
					    WHEN CRSArrTime = 'NA' THEN NULL
                        WHEN CRSArrTime = '2400' THEN '0000'
                        ELSE CRSArrTime 
					END, 
		4, '0' ),'HHmm'), 'HH:mm') as CRSArrTime
    """,        
    "UniqueCarrier",
    "cast(FlightNum as int) as FlightNum",
    "cast(TailNum as STRING) as TailNum",
    "try_cast(ActualElapsedTime as int) as ActualElapsedTime",
    "try_cast(CRSElapsedTime as int) as CRSElapsedTime",
    "try_cast(AirTime as int) as AirTime",
    "try_cast(ArrDelay as int) as ArrDelay",
    "try_cast(DepDelay as int) as DepDelay",
    "Origin",
    "Dest",
    "try_cast(Distance as int) as Distance",
    "try_cast(TaxiIn as int) as TaxiIn",
    "try_cast(TaxiOut as int) as TaxiOut",
    "Cancelled",
    "CancellationCode",
    "try_cast(Diverted as int) as castDiverted",
    "try_cast(CarrierDelay as int) as CarrierDelay",
    "try_cast(WeatherDelay as int) as WeatherDelay",
    "try_cast(NASDelay as int) as NASDelay",
    "try_cast(SecurityDelay as int) as SecurityDelay",
    "try_cast(LateAircraftDelay as int) as LateAircraftDelay",
    "to_date(Date_Part,'yyyy-MM-dd') as Date_Part"
)

display(df_base)

In [0]:

# Writing data
df_base.writeStream.trigger(once=True).format("delta").option(
    "checkpointLocation", "/dbfs/FileStore/tables/checkpointLocation/flight"
).start("/mnt/geeks/cld_adls/flight")

#### Creating delata table on the data


In [0]:
f_delta_cleansed_load('flight', "abfss://cleansed@geekadlsstoragesink.dfs.core.windows.net/flight/", 'cleansed_geekcoders')

In [0]:
%sql

select * from cleansed_geekcoders.flight;